# 01 — Major Dataset Setup and Validation

**Project:** Grounded, Not Just Attending: A Targeted Consistency Objective for Attention-Correct, Semantically Incorrect Image Captioning Errors

## Purpose

This notebook establishes and validates the Major project's dataset interface. It is for **dataset inspection and integrity checks**, not model training.

It will:
1. Load the central Major configuration from `config.py`.
2. Set reproducibility seeds.
3. Inspect configured COCO annotation and image paths.
4. Load COCO caption annotations into a consistent image-ID-based representation.
5. Validate image/annotation correspondence without silently accepting missing data.
6. Display verified examples.
7. Save a dataset audit report.

## Methodological rules

- Do not use the Minor project's simulated saliency maps as training or evaluation evidence.
- Do not assign error scenarios randomly.
- Keep Minor diagnostic annotations separate from Major training data.
- Missing images or annotations must be reported explicitly; do not fabricate replacements.
- Confirm dataset split and annotation availability before official experiments.

**Expected location:** `notebooks/01_setup_and_dataset.ipynb`  
**Config location:** `config.py` at the repository root.

## 1. Imports and environment

Install dependencies from the repository's `requirements.txt`; do not install packages dynamically inside this notebook.

In [ ]:
from pathlib import Path
import json
import random
import platform
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import config as cfg

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", cfg.PROJECT_ROOT)

## 2. Reproducibility

Seed Python and NumPy here. PyTorch/CUDA and DataLoader seeding will be handled by the training and inference utilities.

In [ ]:
SEED = int(cfg.SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Python and NumPy seeds set to {SEED}")

## 3. Inspect configured paths

This is a path report, not a claim that the dataset is installed. The paths in `config.py` must match the actual local directory layout.

In [ ]:
path_items = {
    "project_root": cfg.PROJECT_ROOT,
    "data_dir": cfg.DATA_DIR,
    "coco_dir": cfg.COCO_DIR,
    "annotations_dir": cfg.ANNOTATIONS_DIR,
    "train_images_dir": cfg.TRAIN_IMAGES_DIR,
    "val_images_dir": cfg.VAL_IMAGES_DIR,
    "train_captions": cfg.CAPTIONS_TRAIN_JSON,
    "val_captions": cfg.CAPTIONS_VAL_JSON,
    "train_instances": cfg.INSTANCES_TRAIN_JSON,
    "val_instances": cfg.INSTANCES_VAL_JSON,
}

path_report = pd.DataFrame([
    {"item": name, "path": str(path), "exists": Path(path).exists()}
    for name, path in path_items.items()
])
display(path_report)

## 4. Configuration validation

The config validator creates the configured experiment output directories. Missing dataset files are expected if the dataset has not yet been installed; inspect the report and do not proceed to dataset-dependent cells until required files are present.

In [ ]:
config_checks = cfg.validate_config(verbose=True)
config_check_table = pd.DataFrame(
    [{"check": name, "passed": bool(value)} for name, value in config_checks.items()]
)
display(config_check_table)

## 5. COCO caption annotation loader

The loader preserves COCO image IDs and filenames and groups reference captions by image ID. It does not filter examples based on saliency/fixation files and does not assign synthetic labels.

In [ ]:
def load_coco_caption_annotations(annotation_path: Path):
    """Load COCO captions JSON and return image and caption records."""
    annotation_path = Path(annotation_path)
    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Caption annotation file not found: {annotation_path}\n"
            "Check the dataset paths in config.py before continuing."
        )

    with annotation_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    required_top_level = {"images", "annotations"}
    missing_keys = required_top_level - set(data)
    if missing_keys:
        raise ValueError(
            f"Invalid COCO captions JSON; missing top-level keys: {sorted(missing_keys)}"
        )

    images = {}
    for item in data["images"]:
        if "id" not in item or "file_name" not in item:
            continue
        images[int(item["id"])] = {
            "image_id": int(item["id"]),
            "file_name": str(item["file_name"]),
            "width": item.get("width"),
            "height": item.get("height"),
        }

    captions_by_image = {image_id: [] for image_id in images}
    annotation_rows = []
    for ann in data["annotations"]:
        if "image_id" not in ann or "caption" not in ann:
            continue
        image_id = int(ann["image_id"])
        caption = str(ann["caption"]).strip()
        if not caption:
            continue
        captions_by_image.setdefault(image_id, []).append(caption)
        annotation_rows.append({
            "annotation_id": ann.get("id"),
            "image_id": image_id,
            "caption": caption,
        })

    return images, captions_by_image, pd.DataFrame(annotation_rows)


def summarize_coco_annotations(annotation_path: Path):
    images, captions_by_image, caption_rows = load_coco_caption_annotations(annotation_path)
    caption_counts = [len(captions_by_image.get(image_id, [])) for image_id in images]
    summary = {
        "annotation_path": str(annotation_path),
        "n_images": len(images),
        "n_caption_annotations": len(caption_rows),
        "n_images_with_captions": sum(count > 0 for count in caption_counts),
        "n_images_without_captions": sum(count == 0 for count in caption_counts),
        "mean_captions_per_image": float(np.mean(caption_counts)) if caption_counts else 0.0,
    }
    return summary, images, captions_by_image, caption_rows

## 6. Choose a split to inspect

Start with the development split for a quick setup check. Training will use the configured training split in a later pipeline. Do not use development/test data for training.

If the annotation file is not available locally, this cell stops with an explicit error.

In [ ]:
INSPECT_SPLIT = cfg.DEV_SPLIT
inspect_annotation_path = cfg.get_caption_annotation_path(INSPECT_SPLIT)
inspect_image_dir = cfg.get_image_dir(INSPECT_SPLIT)

dataset_summary, images_by_id, captions_by_image, caption_rows = summarize_coco_annotations(
    inspect_annotation_path
)

display(pd.DataFrame([dataset_summary]))
print("Image directory:", inspect_image_dir)
print("Image directory exists:", inspect_image_dir.exists())

## 7. Validate image files

This uses the `file_name` values in the COCO JSON rather than assuming filenames can always be reconstructed from IDs. Missing files are reported; no fallback images are fabricated.

For a full split, this scan may take some time.

In [ ]:
def validate_image_files(
    images_by_id: dict,
    image_dir: Path,
    captions_by_image: dict,
    max_images=None,
):
    records = list(images_by_id.values())
    if max_images is not None:
        records = records[:int(max_images)]

    results = []
    for item in records:
        image_path = Path(image_dir) / item["file_name"]
        exists = image_path.is_file()
        results.append({
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "image_path": str(image_path),
            "exists": exists,
            "has_caption": len(captions_by_image.get(item["image_id"], [])) > 0,
        })
    return pd.DataFrame(results)


image_check_limit = cfg.DEBUG_MAX_IMAGES if cfg.DEBUG_MODE else None
image_validation = validate_image_files(
    images_by_id,
    inspect_image_dir,
    captions_by_image,
    max_images=image_check_limit,
)

n_missing = int((~image_validation["exists"]).sum()) if len(image_validation) else 0
n_present = int(image_validation["exists"].sum()) if len(image_validation) else 0

print(f"Checked: {len(image_validation)}")
print(f"Present: {n_present}")
print(f"Missing: {n_missing}")

display(image_validation.head(10))

if n_missing:
    print("WARNING: Missing image files were found. Review the report before using this split.")

## 8. Dataset integrity summary

These checks describe annotation/image consistency for the inspected split. A successful JSON load alone does not mean the dataset is ready for official experiments.

In [ ]:
integrity_summary = {
    "split": INSPECT_SPLIT,
    "annotation_file_exists": inspect_annotation_path.exists(),
    "image_directory_exists": inspect_image_dir.exists(),
    "annotation_image_count": len(images_by_id),
    "caption_annotation_count": len(caption_rows),
    "images_checked": len(image_validation),
    "images_present": n_present,
    "images_missing": n_missing,
    "images_with_no_caption": int(
        sum(len(captions_by_image.get(image_id, [])) == 0 for image_id in images_by_id)
    ),
}
display(pd.DataFrame([integrity_summary]))

## 9. Inspect examples

Display only examples whose image files exist. This is a visual sanity check of the local dataset, not a model result.

In [ ]:
def show_dataset_examples(
    image_validation: pd.DataFrame,
    captions_by_image: dict,
    n_examples: int = 4,
):
    available = image_validation[image_validation["exists"]].head(n_examples)
    if available.empty:
        print("No existing images available to display.")
        return

    fig, axes = plt.subplots(len(available), 1, figsize=(10, 4 * len(available)))
    if len(available) == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, available.iterrows()):
        path = Path(row["image_path"])
        with Image.open(path) as im:
            ax.imshow(im.convert("RGB"))
        ax.axis("off")

        captions = captions_by_image.get(int(row["image_id"]), [])
        caption_preview = "\n".join(f"• {caption}" for caption in captions[:3])
        ax.set_title(
            f"image_id={row['image_id']} | {row['file_name']}\n{caption_preview}",
            fontsize=10,
            loc="left",
        )

    plt.tight_layout()
    plt.show()


show_dataset_examples(image_validation, captions_by_image, n_examples=4)

## 10. Optional deterministic development sample

For smoke tests, create a small deterministic list of available image IDs. This is a development convenience only; official split/sample choices must be recorded in the experiment manifest.

In [ ]:
available_image_ids = image_validation.loc[
    image_validation["exists"], "image_id"
].astype(int).tolist()

rng = np.random.default_rng(SEED)
sample_size = min(10, len(available_image_ids))
debug_image_ids = sorted(
    rng.choice(available_image_ids, size=sample_size, replace=False).tolist()
) if sample_size else []

print(f"Deterministic development sample ({len(debug_image_ids)} IDs):")
print(debug_image_ids)

## 11. Save dataset audit report

The report records paths, counts, split, and seed. It is written under the Major experiment outputs.

In [ ]:
audit_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(cfg.PROJECT_ROOT),
    "dataset_name": cfg.DATASET_NAME,
    "split": INSPECT_SPLIT,
    "seed": SEED,
    "annotation_path": str(inspect_annotation_path),
    "image_directory": str(inspect_image_dir),
    "dataset_summary": dataset_summary,
    "integrity_summary": integrity_summary,
    "debug_image_ids": debug_image_ids,
    "notes": [
        "This is a dataset setup audit, not a training run.",
        "No random scenario labels were assigned.",
        "No saliency/fixation fallback or simulated visual evidence was used.",
        "Missing image files are reported, not replaced.",
    ],
}

audit_dir = cfg.OUTPUT_DIR / "dataset_audit"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_json_path = audit_dir / f"{INSPECT_SPLIT}_dataset_audit.json"
audit_csv_path = audit_dir / f"{INSPECT_SPLIT}_image_validation.csv"

with audit_json_path.open("w", encoding="utf-8") as f:
    json.dump(audit_report, f, indent=2)

image_validation.to_csv(audit_csv_path, index=False)

print("Saved audit report:", audit_json_path)
print("Saved image validation table:", audit_csv_path)

## 12. Readiness checklist

Review before declaring dataset setup complete:

- [ ] COCO image and caption annotation paths are correct.
- [ ] Training and development splits are separate.
- [ ] Test data is not used for training or model selection.
- [ ] Missing image files have been investigated.
- [ ] Subset limits are documented and deterministic.
- [ ] Minor diagnostic annotations remain separate from Major training data.
- [ ] No simulated saliency or randomly assigned error categories enter the pipeline.
- [ ] Dataset protocol is recorded before official experiments.

## Outcome

This notebook establishes the dataset interface and integrity checks. It does **not** implement BLIP inference, token eligibility, region extraction, consistency loss, or training; those belong in later modules after the corresponding audits.